### Retrain Crop-Based Classifiers

Fine-tunes the existing binary, 5-class, and/or 4-class InsectNet classifiers with newly labeled crops.
Faster and needs fewer new samples than training from scratch.
Old weights are **backed up** to `outputs/training/model_runs/` before overwriting.

**Workflow:** label crops → move to `annotated_crops/` → **run this notebook**

**Input** — `data/training/annotated_crops/{class}/` · `models/binary_best.pth` · `models/5group_*.pth` · `models/4group_insectnet.pth`  
**Output** — `outputs/training/model_runs/{RUN_NAME}_{ts}/` (logs + backup) · **overwrites** `models/binary_best.pth`, `models/5group_*.pth`, and/or `models/4group_insectnet.pth`

**Must edit (Cell 2):**

| Variable | What it controls |
|----------|-----------------|
| `RETRAIN_BINARY` | Set False to skip binary classifier |
| `RETRAIN_5CLASS_EFFNET` / `RETRAIN_5CLASS_INSECTNET` | Set False to skip each 5-class model |
| `RETRAIN_4CLASS` | Set False to skip 4-class InsectNet |
| `IMG_SIZE_BIN` | Must match `binary_best.pth` training size (224) — shape mismatch crash otherwise |
| `IMG_SIZE_5CLS_EN` / `IMG_SIZE_5CLS_IN` | Must match respective 5-class model training size (224) |
| `IMG_SIZE_4CLS` | Must match `4group_insectnet.pth` training size (224) |

**Optional (Cell 2):** `LR_FINETUNE` (1e-4 — keep lower than original 1e-3), `EPOCHS_BINARY` / `EPOCHS_5CLS_EN` / `EPOCHS_5CLS_IN` (12), `FREEZE_BACKBONE` (True — set False for full fine-tune with large new dataset), `BATCH` (32), `DATASETS` ([] = all sub-folders)

**Background sampling:** balanced across camera plots — each plot contributes an equal quota so busy plots cannot dominate the training set. Both filename formats are handled automatically:
- **Current:** `Site_species_plot_date__Camera__Image_crop.jpg`
- **Legacy HDD:** `hdd_N_year_site_species_plot[_date]_NNN_WSCT__Image_crop.jpg` — camera-overflow folders (`_101_WSCT`, `_102_WSCT`, …) are the same physical camera hitting the 9 999-image folder limit and are merged into one plot key automatically.

If the filename format changes in a future season, update only `parse_plot_key()` — see its docstring.


##### Cell 1 — Environment  *(no edits needed)*

**Local:** auto-detects the repo root via `git rev-parse --show-toplevel` — no path editing required.

**Colab:** uses the path extracted from the zip in Cell 0.

Sets all derived paths (`MODEL_DIR`, `LABELED_DIR`, output folders).

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import zipfile, os
    DRIVE_ROOT = Path('/content/drive/MyDrive')
    ZIP_PATH   = DRIVE_ROOT / 'pollinator-colab.zip'
    EXTRACT_TO = Path('/content/pollinator-colab')
    if not EXTRACT_TO.exists():
        print(f'Extracting {ZIP_PATH.name} ...')
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall('/content/')
        print('✓ Extracted to /content/pollinator-colab')
    else:
        print('✓ Already extracted')
    BASE_DIR   = EXTRACT_TO
    DRIVE_BASE = DRIVE_ROOT / 'pollinator-colab'
else:
    import subprocess as _sp
    _git_root  = Path(_sp.check_output(
        ['git', 'rev-parse', '--show-toplevel'], text=True).strip())
    BASE_DIR   = _git_root / 'ml_pipelines' / 'notebooks' / 'pollinator_detection'
    DRIVE_BASE = BASE_DIR

MODEL_DIR   = BASE_DIR / 'models'
LABELED_DIR = BASE_DIR / 'data' / 'training' / 'annotated_crops'
INSECTNET_W = BASE_DIR / 'InsectNet' / 'model.pth'
# Training outputs go to local SSD on Colab (fast); saved to Drive after training.
LOCAL_TRAINING = Path('/content/outputs/training') if IN_COLAB else BASE_DIR / 'outputs' / 'training'
LOCAL_TRAINING.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f'Env        : {"Colab" if IN_COLAB else "Local"}')
print(f'BASE_DIR   : {BASE_DIR}  exists={BASE_DIR.exists()}')
print(f'MODEL_DIR  : {MODEL_DIR}  exists={MODEL_DIR.exists()}')
print(f'LABELED_DIR: {LABELED_DIR}  exists={LABELED_DIR.exists()}')
print(f'INSECTNET_W: {INSECTNET_W}  exists={INSECTNET_W.exists()}')

def resolve_web_dirs(web_root, batches):
    """Return batch dirs to load web images from.

    web_root : Path to data/training/web_images/
    batches  : list of batch folder names, e.g. ['batch_20260529_143000']
               Empty list [] = all batch_*/ folders (default).
               Falls back to web_root itself if no batch_*/ folders exist
               (legacy flat layout).
    """
    all_batches = sorted(
        d for d in Path(web_root).iterdir()
        if d.is_dir() and d.name.startswith('batch_')
    ) if Path(web_root).exists() else []
    if not all_batches:
        return [Path(web_root)] if Path(web_root).exists() else []
    if batches:
        return [Path(web_root) / b for b in batches]
    return all_batches


##### Cell 2 — Retrain config  ← **edit before retraining**

Key parameters:

- **`DATASETS`** — which sub-folders of `annotated_crops/` to include.
  Leave `[]` to use **all** sub-folders automatically.
  Set explicitly when you only want to retrain on newly labeled data, e.g.:
  ```
  DATASETS = ['labeled_new_run']   # only new crops
  DATASETS = ['labeled_ls', 'labeled_mb']  # specific sets
  ```
  **Check the Cell 5 data summary before training** — it prints per-class counts
  so you can confirm each class has enough samples (aim for ≥ 50 per class).
  Classes with very few crops (e.g. bumblebee < 20) will produce unreliable metrics.

- **`RETRAIN_BINARY`** / **`RETRAIN_5CLASS_EFFNET`** / **`RETRAIN_5CLASS_INSECTNET`** / **`RETRAIN_4CLASS`** —
  toggle which models to update. Disable models you don't need to speed up the run.

- **`EPOCHS_BINARY`** / **`EPOCHS_5CLS_EN`** / **`EPOCHS_4CLASS`** / **`EPOCHS_5CLS_IN`** — training epochs.
  Fine-tuning typically needs fewer epochs than training from scratch. Start with 10–15.

- **`LR_FINETUNE`** — use a **smaller** value than initial training (1e-4 instead of
  1e-3) to avoid overwriting the pretrained features.

- **`FREEZE_BACKBONE`** — if True, only the classifier head is updated. Recommended
  when new data is small (< ~200 new crops). Set False for larger updates.

- **`BG_RATIO`** — background crops per insect crop in binary training.

- **Web images** — `USE_WEB_FOR_BINARY = True` adds iNaturalist images to the binary
  insect class. The 4-class and 5-class retrains use **field crops only**
  (no web images) — this is intentional so the group classifier stays calibrated
  to camera-trap crop appearance.


In [ ]:
# ── Which sub-datasets to use ──────────────────────────────────────────
# Sub-folder names inside data/training/annotated_crops/.
# [] = ALL sub-folders (full retrain on everything).
# List specific folders to retrain on new data only, e.g.:
#   DATASETS = ['labeled_new_run']          # only new crops
#   DATASETS = ['labeled_ls', 'labeled_mb'] # two specific sets
# Check Cell 5 printout for per-class crop counts before running.
DATASETS = []   # ← edit this

# ── Which models to retrain ──────────────────────────────────────────
RETRAIN_BINARY  = True
RETRAIN_5CLASS_EFFNET  = True   # 5-class EfficientNet-B2
RETRAIN_4CLASS           = True   # 4-class InsectNet group classifier
RETRAIN_5CLASS_INSECTNET = True   # 5-class InsectNet group classifier

# ── Hyperparameters ───────────────────────────────────────────────
EPOCHS_BINARY   = 12     # epochs for binary classifier fine-tune
EPOCHS_5CLS_EN  = 12     # epochs for 5-class EfficientNet fine-tune
EPOCHS_4CLASS   = 12     # epochs for 4-class InsectNet fine-tune
EPOCHS_5CLS_IN  = 12     # epochs for 5-class InsectNet fine-tune
LR_FINETUNE     = 1e-4   # lower LR for fine-tuning (was 1e-3 for initial training)
BATCH           = 32
IMG_SIZE_BIN    = 224    # must match binary_best.pth (EfficientNet-B2 binary, trained at 224)
IMG_SIZE_5CLS_EN = 224   # must match 5group_efficientnet.pth (EfficientNet-B2, trained at 224)
IMG_SIZE_4CLS   = 224    # must match 4group_insectnet.pth (RegNet-Y-32GF, trained at 224)
IMG_SIZE_5CLS_IN = 224   # must match 5group_insectnet.pth
BG_RATIO        = 3      # background:insect ratio for binary training
SEED            = 42

# ── Freeze backbone? ────────────────────────────────────────────────
# True  = only update the classifier head (faster, safer for small datasets)
# False = update the entire network (recommended if > ~500 new crops)
FREEZE_BACKBONE = False


# ── Which checkpoint to fine-tune from ──────────────────────────────
# Default: current production model in models/.
# Change to a specific run to fine-tune from a different checkpoint, e.g.:
#   BINARY_MODEL = LOCAL_TRAINING / 'model_runs' / 'retrain_20250529_140000' / 'binary_best.pth'
BINARY_MODEL = MODEL_DIR / 'binary_best.pth'          # EfficientNet-B2 binary
MODEL_5CLASS_EFFNET = MODEL_DIR / '5group_efficientnet.pth'  # EfficientNet-B2 5-class
MODEL_4CLASS           = MODEL_DIR / '4group_insectnet.pth'     # InsectNet (RegNet-Y-32GF) 4-class
MODEL_5CLASS_INSECTNET = MODEL_DIR / '5group_insectnet.pth'     # InsectNet (RegNet-Y-32GF) 5-class

# ── Class definitions ────────────────────────────────────────────────
CLASSES_BINARY  = ['background', 'insect']
CLASSES_5       = ['bumblebee', 'fly', 'butterfly', 'other', 'background']
INSECT_FOLDERS      = ['bumblebee', 'fly', 'butterfly', 'other']

# ── Web images for binary insect class ──────────────────────────────
WEB_ROOT           = BASE_DIR / 'data' / 'training' / 'web_images'  # iNaturalist images root
# Which batch folders to use ([] = all batches, e.g. ['batch_20260529_143000'] = one batch)
WEB_BATCHES        = []   # ← change to restrict which download rounds are used for training
USE_WEB_FOR_BINARY = True   # add web images as extra insect data for binary classifier

# Folder name -> canonical class (handles legacy naming)
ALIAS_5 = {
    'bumblebee':     'bumblebee',
    'fly':           'fly',
    'butterfly':     'butterfly',
    'butterfly_moth':  'butterfly',
    'other':         'other',
    'background':    'background',
}

print('Config ready.')
print(f'  Models  : binary={BINARY_MODEL.name}  5cls-EN={MODEL_5CLASS_EFFNET.name}  5cls-IN={MODEL_5CLASS_INSECTNET.name}  4cls={MODEL_4CLASS.name}')
print(f'  Binary  : retrain={RETRAIN_BINARY}  epochs={EPOCHS_BINARY}  lr={LR_FINETUNE}  freeze={FREEZE_BACKBONE}')
print(f'  5cls-EN : retrain={RETRAIN_5CLASS_EFFNET}  epochs={EPOCHS_5CLS_EN}  lr={LR_FINETUNE}  freeze={FREEZE_BACKBONE}')
print(f'  4-class : retrain={RETRAIN_4CLASS}  epochs={EPOCHS_4CLASS}  lr={LR_FINETUNE}  freeze={FREEZE_BACKBONE}')
print(f'  5cls-IN : retrain={RETRAIN_5CLASS_INSECTNET}  epochs={EPOCHS_5CLS_IN}  lr={LR_FINETUNE}  freeze={FREEZE_BACKBONE}')
# ── Resolve dataset sub-folders ─────────────────────────────────
if DATASETS:
    DATASET_DIRS = [LABELED_DIR / ds for ds in DATASETS]
else:
    DATASET_DIRS = sorted([d for d in LABELED_DIR.iterdir()
                           if d.is_dir() and d.name != 'progress'])
print(f'Datasets : {[d.name for d in DATASET_DIRS]}')


# ── Run output directory ─────────────────────────────────────────────────────
from datetime import datetime as _dt
_ts    = _dt.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = LOCAL_TRAINING / 'model_runs' / f'retrain_{_ts}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f'Run dir    : {RUN_DIR}')


##### Cell 3 — Imports + training utilities

Loads PyTorch, torchvision, and all shared training functions. **Do not edit.**

In [ ]:
import shutil
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict
from PIL import Image
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
try:
    from sklearn.metrics import classification_report
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False

DEVICE = (torch.device('cuda') if torch.cuda.is_available()
          else torch.device('mps') if torch.backends.mps.is_available()
          else torch.device('cpu'))
print(f'Device  : {DEVICE}  |  PyTorch: {torch.__version__}')
if torch.cuda.is_available(): print(f'GPU     : {torch.cuda.get_device_name(0)}')


def letterbox(img, size):
    w, h = img.size; ms = max(w, h)
    sq = Image.new('RGB', (ms, ms), (0, 0, 0))
    sq.paste(img, ((ms - w) // 2, (ms - h) // 2))
    return sq.resize((size, size), Image.BILINEAR)

class CropDataset(Dataset):
    def __init__(self, samples, tf): self.s = samples; self.tf = tf
    def __len__(self): return len(self.s)
    def __getitem__(self, i):
        p, l = self.s[i]; return self.tf(Image.open(p).convert('RGB')), l

class _Letterbox:
    """Picklable letterbox transform."""
    def __init__(self, size): self.size = size
    def __call__(self, img): return letterbox(img, self.size)

def make_tf(sz, aug=False):
    base = [_Letterbox(sz), T.ToTensor(),
             T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]
    if aug:
        base = [
            _Letterbox(sz),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.RandomRotation(degrees=30),
            T.ColorJitter(brightness=0.4, contrast=0.4,
                          saturation=0.3, hue=0.08),
            T.RandomGrayscale(p=0.05),
            T.RandomAffine(degrees=0, translate=(0.1, 0.1),
                           scale=(0.85, 1.15)),
        ] + base[1:]
    return T.Compose(base)

def make_loader(samples, idxs, sz, batch, aug=False, weighted=True):
    sub = [samples[i] for i in idxs]; ds = CropDataset(sub, make_tf(sz, aug))
    if weighted and sub:
        labs = [s[1] for s in sub]; cnt = np.bincount(labs, minlength=max(labs)+1)
        wts = [1.0/max(1,cnt[l]) for l in labs]
        return DataLoader(ds, batch_size=batch,
                          sampler=WeightedRandomSampler(wts, len(wts)), num_workers=0, pin_memory=torch.cuda.is_available())
    return DataLoader(ds, batch_size=batch, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

def stratified_split(samples, val_frac=0.1, test_frac=0.1, seed=42):
    rng = np.random.default_rng(seed); by_cls = defaultdict(list)
    for i, (_, l) in enumerate(samples): by_cls[l].append(i)
    tr, va, te = [], [], []
    for l, idxs in by_cls.items():
        rng.shuffle(idxs); n = len(idxs)
        nva = max(1, int(n*val_frac)); nte = max(1, int(n*test_frac))
        tr.extend(idxs[nva+nte:]); va.extend(idxs[:nva]); te.extend(idxs[nva:nva+nte])
    return tr, va, te

def collect_crops(dataset_dirs, classes, alias):
    """dataset_dirs: list of Path, each containing class subfolders."""
    ci = {c:i for i,c in enumerate(classes)}; smp = []
    for labeled_dir in dataset_dirs:
        for folder, cls in alias.items():
            if cls not in ci: continue
            dd = Path(labeled_dir)/folder
            if not dd.exists(): continue
            for ext in ('*.jpg','*.jpeg','*.png'):
                for p in dd.glob(ext): smp.append((p, ci[cls]))
    counts = {i:0 for i in range(len(classes))}
    for _,l in smp: counts[l] += 1
    return smp, counts

def weighted_criterion(counts, n, device):
    w = torch.tensor([1.0/max(1,counts.get(i,1)) for i in range(n)], dtype=torch.float, device=device)
    return nn.CrossEntropyLoss(weight=w/w.sum())

@torch.no_grad()
def eval_epoch(model, loader, criterion, device, classes):
    model.eval(); ls=cor=tot=0; ap,al=[],[]
    tp={i:0 for i in range(len(classes))}; fp={i:0 for i in range(len(classes))}; fn={i:0 for i in range(len(classes))}
    for imgs,labels in loader:
        imgs,labels=imgs.to(device),labels.to(device); out=model(imgs); preds=out.argmax(1)
        ls+=criterion(out,labels).item()*labels.size(0)
        cor+=(preds==labels).sum().item(); tot+=labels.size(0)
        ap.extend(preds.cpu().tolist()); al.extend(labels.cpu().tolist())
        for i in range(len(classes)):
            tp[i]+=((preds==i)&(labels==i)).sum().item()
            fp[i]+=((preds==i)&(labels!=i)).sum().item()
            fn[i]+=((preds!=i)&(labels==i)).sum().item()
    pf={}; pr={}
    for i in range(len(classes)):
        p2=tp[i]/max(1,tp[i]+fp[i]); r=tp[i]/max(1,tp[i]+fn[i])
        pf[classes[i]]=2*p2*r/max(1e-8,p2+r); pr[classes[i]]=r
    return {'loss':ls/tot,'acc':cor/tot,'macro_f1':sum(pf.values())/max(1,len(classes)),
            'per_f1':pf,'per_recall':pr,'preds':ap,'labels':al}

def load_efficientnet_for_finetune(ckpt_path, n_classes, freeze_backbone):
    model = torchvision.models.efficientnet_b2(weights='IMAGENET1K_V1')
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, n_classes)
    if Path(ckpt_path).exists():
        ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        saved_classes = ckpt.get('classes', [])
        if len(saved_classes) == n_classes:
            model.load_state_dict(ckpt['state_dict'])
            print(f'  Loaded: {Path(ckpt_path).name}  '
                  f'classes={saved_classes}  val_f1={ckpt.get("val_macro_f1", "?")}')
        else:
            print(f'  WARNING: checkpoint has {len(saved_classes)} classes but model '
                  f'expects {n_classes}. Starting from ImageNet weights.')
    else:
        print(f'  No checkpoint at {ckpt_path}. Starting from ImageNet weights.')
    if freeze_backbone:
        for name, param in model.named_parameters():
            param.requires_grad = name.startswith('classifier')
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'  Backbone frozen — {trainable:,} trainable params (head only)')
    else:
        print(f'  Full network — {sum(p.numel() for p in model.parameters()):,} trainable params')
    return model

def run_finetune(model, name, tr_ldr, va_ldr, epochs, lr, counts, ckpt, device, classes, model_dir, in_colab, best_metric='macro_f1', img_size=224):
    crit=weighted_criterion(counts,len(classes),device)
    opt=torch.optim.Adam(filter(lambda p:p.requires_grad,model.parameters()),lr=lr)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    best_f1=0.0; hist={'tr':[],'va':[],'f1':[]}
    score_label = 'InsRecall' if best_metric == 'recall_insect' else 'Score    '
    hdr = (f'{"Ep":>4}  {"TrLoss":>8}  {"VaLoss":>8}  {"MacroF1":>8}  {score_label:>9}  {"Acc":>6}'
           + ''.join(f'  {c[:7]:>8}_F1  {c[:7]:>8}_Rec' for c in classes))
    print(f'\n{"="*80}\n{name}  epochs={epochs}  lr={lr}  best={best_metric}\n{"="*80}\n{hdr}')
    for ep in range(1,epochs+1):
        model.train(); tl=tc=tt=0
        for imgs,labels in tr_ldr:
            imgs,labels=imgs.to(device),labels.to(device)
            opt.zero_grad(); out=model(imgs); loss=crit(out,labels)
            loss.backward(); opt.step()
            tl+=loss.item()*labels.size(0); tc+=(out.argmax(1)==labels).sum().item(); tt+=labels.size(0)
        vr=eval_epoch(model,va_ldr,crit,device,classes); sched.step()
        _score = (vr['per_recall'].get('insect', 0) if best_metric == 'recall_insect'
                  else vr['macro_f1'])
        new_best=_score>best_f1
        if new_best:
            best_f1=_score
            torch.save({'state_dict':model.state_dict(),'classes':classes,
                        'img_size':img_size,'val_macro_f1':vr['macro_f1'],'epoch':ep},ckpt)
        pf=vr['per_f1']
        _per = ''.join(
            f'  {pf.get(c,0):>10.3f}  {vr["per_recall"].get(c,0):>10.3f}'
            for c in classes)
        print(f'{ep:>4}  {tl/tt:>8.4f}  {vr["loss"]:>8.4f}  {vr["macro_f1"]:>8.3f}  {_score:>9.3f}  {vr["acc"]:>6.3f}'
              + _per + ('  *' if new_best else ''))
        hist['tr'].append(tl/tt); hist['va'].append(vr['loss']); hist['f1'].append(vr['macro_f1'])
    print(f'\nBest val {best_metric}: {best_f1:.3f}  ->  {ckpt}')
    if in_colab:
        try:
            _drive_models = Path('/content/drive/MyDrive/pollinator-colab/models')
            _drive_models.mkdir(parents=True, exist_ok=True)
            shutil.copy(str(ckpt), str(_drive_models / Path(ckpt).name))
            print(f'Drive backup ok → {_drive_models / Path(ckpt).name}')
        except Exception as e: print(f'Drive backup failed: {e}')
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,4))
    ax1.plot(hist['tr'],label='train'); ax1.plot(hist['va'],label='val')
    ax1.set_title('Loss'); ax1.legend(); ax1.grid(True)
    ax2.plot(hist['f1'],lw=2,label='macro F1'); ax2.set_title('Macro F1'); ax2.grid(True)
    plt.suptitle(name); plt.tight_layout()
    curves_path=Path(model_dir)/f'{name}_retrain_curves.png'
    plt.savefig(curves_path,dpi=100); plt.close(); print(f'Curves: {curves_path}')
    return model


def load_insectnet_for_finetune(ckpt_path, n_classes, freeze_backbone):
    """Load existing 4group_insectnet.pth for fine-tuning."""
    model = torchvision.models.regnet_y_32gf()
    model.fc = nn.Linear(3712, n_classes)
    if Path(ckpt_path).exists():
        ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        saved_classes = ckpt.get('classes', [])
        if len(saved_classes) == n_classes:
            model.load_state_dict(ckpt['state_dict'])
            print(f'  Loaded: {Path(ckpt_path).name}  '
                  f'classes={saved_classes}  val_f1={ckpt.get("val_macro_f1", "?")}')
        else:
            print(f'  WARNING: checkpoint has {len(saved_classes)} classes, '
                  f'model expects {n_classes}. Check checkpoint.')
    else:
        print(f'  No checkpoint at {ckpt_path}. Cannot retrain without base model.')
    # Freeze all, then unfreeze fc (and block4 if not freeze_backbone)
    for param in model.parameters():
        param.requires_grad = False
    for name, param in model.named_parameters():
        if name.startswith('fc.'):
            param.requires_grad = True
        if not freeze_backbone and 'trunk_output.block4' in name:
            param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    mode = 'fc only' if freeze_backbone else 'block4 + fc'
    print(f'  Trainable ({mode}): {trainable:,} params')
    return model

print('\u2713 Utilities loaded.')

def parse_plot_key(path):
    """
    Extract a per-camera-plot group key from a crop filename for stratified
    background sampling. Returns a string identifying one plot (site+species+plot+date).

    CURRENT FORMAT: <site>__<camera>__<image>_crop<i>.jpg
      e.g. Gruvan_Bal_p1_20250727__102_WSCT__WSCT1529_crop00.jpg
      Returns: Gruvan_Bal_p1_20250727

    LEGACY FORMAT (hdd_*):
      hdd_<n>_<year>_<site>_<species>_<plot>[_<date>]_<NNN>_WSCT__<image>.jpg
      The _101_WSCT/_102_WSCT/... suffix is the SAME camera overflowing at
      9999 images into a new folder. We strip this so all overflow folders
      share one plot key.
      e.g. hdd_2_2025_desert_Vau_p1_20250725_101_WSCT__...
           hdd_2_2025_desert_Vau_p1_20250725_102_WSCT__...
           both return: hdd_2_2025_desert_Vau_p1_20250725
    """
    import re as _re
    stem = Path(path).stem
    # Both formats use __ as separator — take everything before the first __
    base = stem.split('__')[0] if '__' in stem else stem
    # Strip accidental whitespace in folder names
    base = base.strip()
    # Strip trailing camera-overflow suffix _NNN_WSCT (case-insensitive)
    base = _re.sub(r'_\d+_[Ww][Ss][Cc][Tt]$', '', base)
    return base if base else 'unknown'
def sample_bg(bg_paths, n_total, seed=42):
    """
    Sample n_total background crops balanced across camera plots.

    Uses parse_plot_key() to group crops by plot, then gives each plot an
    equal quota so that no single busy plot can dominate the training set.

    Args:
        bg_paths : list of Path — all available background crop paths
        n_total  : int — how many to sample in total
        seed     : int — RNG seed for reproducibility

    Prints a per-group breakdown so you can verify the balance.
    If a plot has fewer crops than its quota, it contributes all it has
    (the total sampled may be slightly below n_total in that case).
    """
    if not n_total:
        return []

    groups = {}
    for p in bg_paths:
        groups.setdefault(parse_plot_key(p), []).append(p)

    rng = np.random.default_rng(seed)
    for imgs in groups.values():
        rng.shuffle(imgs)

    n_groups  = len(groups)
    quota     = n_total // n_groups
    remainder = n_total  % n_groups

    print(f'Background sampling: {n_groups} plot group(s), quota={quota}/group')
    for key, imgs in sorted(groups.items()):
        print(f'  {key:30}: {len(imgs):>5} available')

    sampled = []
    for i, key in enumerate(sorted(groups)):
        take = quota + (1 if i < remainder else 0)
        sampled.extend(groups[key][:min(take, len(groups[key]))])

    rng.shuffle(sampled)
    print(f'Sampled {len(sampled)} background crops (target {n_total})')
    return sampled

##### Cell 4 — Inspect labeled data

Shows crop counts per class in `annotated_crops/`. The classifier needs at least ~50
crops per class to fine-tune reliably. If a class shows ⚠ low, add more crops via
`crop_labeler.py` before retraining.

In [ ]:
all_cls = ['background', 'bumblebee', 'fly', 'butterfly', 'other', 'unsure']
totals  = {cls: 0 for cls in all_cls}

print(f'{"Dataset":<15}  ' + '  '.join(f'{c[:6]:>6}' for c in all_cls))
print('-' * 75)
for ds_dir in DATASET_DIRS:
    row = []
    for cls in all_cls:
        d = ds_dir / cls
        n = sum(len(list(d.glob(ext))) for ext in ('*.jpg','*.jpeg','*.png')) if d.exists() else 0
        totals[cls] += n
        row.append(n)
    print(f'{ds_dir.name:<15}  ' + '  '.join(f'{n:>6}' for n in row))
print('-' * 75)
print(f'{"TOTAL":<15}  ' + '  '.join(f'{totals[c]:>6}' for c in all_cls))
print()
for cls in ['bumblebee','fly','butterfly','other']:
    t = totals[cls]
    status = '⚠ low (<30)' if t < 30 else ('ok' if t < 100 else 'good')
    print(f'  {cls:<15}: {t:>5}  {status}')


##### Cell 5 — Collect and split data

Loads all labeled crops from `annotated_crops/` and creates stratified train/val/test splits.

**Background sampling** is balanced across camera plots using `parse_plot_key()` + `sample_bg()`:
each plot contributes an equal quota so a busy plot cannot dominate the training set.
The bg quota is based on **field insect crop count only** (not web images).

> **If your crop filenames change** (new field season, different camera naming), update
> `parse_plot_key()` — that is the only function you need to touch.
>
> Supported formats out of the box:
> - `Site_species_plot_date__Camera__Image_crop.jpg` (current)
> - `hdd_N_year_site_species_plot[_date]_NNN_WSCT__Image_crop.jpg` (legacy HDD)
>   Camera-overflow folders (`_101_WSCT`, `_102_WSCT`, …) are automatically
>   merged — they share the same plot quota.


In [ ]:
# Binary data
insect_paths = []
bg_paths_all = []
for ds_dir in DATASET_DIRS:
    for folder in INSECT_FOLDERS:
        for ext in ('*.jpg','*.jpeg','*.png'): insect_paths.extend((ds_dir/folder).glob(ext))
    for ext in ('*.jpg','*.jpeg','*.png'): bg_paths_all.extend((ds_dir/'background').glob(ext))
_field_insect_n = len(insect_paths)  # field crops only — used for bg quota

# ── Add web images to binary insect class ────────────────────────
if USE_WEB_FOR_BINARY and WEB_ROOT and resolve_web_dirs(WEB_ROOT, WEB_BATCHES):
    _web_insect = []
    for _folder in INSECT_FOLDERS:
        for _batch_dir in resolve_web_dirs(WEB_ROOT, WEB_BATCHES):
            for _ext in ('*.jpg','*.jpeg','*.png'):
                _web_insect.extend((_batch_dir / _folder).glob(_ext))
else:
    _web_insect = []  # define even when not used — avoids NameError below
    print('USE_WEB_FOR_BINARY=False or no web batch dirs found — binary uses field crops only')

# bg quota based on field crops only — web images inflate insect count
# but bg crops come from field data, so ratio should match field distribution
bg_sampled  = sample_bg(bg_paths_all, _field_insect_n * BG_RATIO, SEED)
binary_data = ([(p,1) for p in insect_paths]
             + [(p,1) for p in _web_insect]
             + [(p,0) for p in bg_sampled])
bin_counts  = {0: len(bg_sampled), 1: len(insect_paths) + len(_web_insect)}
bin_tr, bin_va, bin_te = stratified_split(binary_data, seed=SEED)
print('Binary dataset:')
print(f'  insect (field) : {len(insect_paths):>5}')
print(f'  insect (web)   : {len(_web_insect):>5}')
print(f'  background     : {bin_counts[0]:>5}  (from {len(bg_paths_all)} available)')
print(f'  split          : train={len(bin_tr)}  val={len(bin_va)}  test={len(bin_te)}')

# 5-class data (field crops only — web images not used in group/5-class retrain)
smp_5, counts_5 = collect_crops(DATASET_DIRS, CLASSES_5, ALIAS_5)
tr_5, va_5, te_5 = stratified_split(smp_5, seed=SEED)
print('\n5-class dataset (field crops only):')
for i, c in enumerate(CLASSES_5): print(f'  {c:20}: {counts_5.get(i,0):>5}')
print(f'  split      : train={len(tr_5)}  val={len(va_5)}  test={len(te_5)}')

# 4-class data (same insect crops, no background)
_CLASSES_4_PRINT = ['bumblebee', 'fly', 'butterfly', 'other']
_ALIAS_4_PRINT   = {'bumblebee':'bumblebee','fly':'fly','butterfly':'butterfly',
                    'butterfly_moth':'butterfly','other':'other'}
smp_4_preview, counts_4_preview = collect_crops(DATASET_DIRS, _CLASSES_4_PRINT, _ALIAS_4_PRINT)
print('\n4-class dataset (field crops only, no background):')
for i, c in enumerate(_CLASSES_4_PRINT): print(f'  {c:20}: {counts_4_preview.get(i,0):>5}')
print(f'  total      : {len(smp_4_preview):>5}')

# ── Low-crop warnings ───────────────────────────────────────────────────
_WARN_THRESHOLD = 50
_warn_classes = [(c, counts_5.get(i, 0)) for i, c in enumerate(CLASSES_5)
                 if c != 'background' and counts_5.get(i, 0) < _WARN_THRESHOLD]
if _warn_classes:
    print('\n⚠  LOW CROP COUNT — these classes may produce unreliable metrics:')
    for _c, _n in _warn_classes:
        print(f'   {_c}: {_n} crops  (aim for ≥ {_WARN_THRESHOLD})')
    print('   → label more crops with crop_labeler.py, or set DATASETS to include more sub-folders.')
else:
    print('\n✓ All insect classes have ≥', _WARN_THRESHOLD, 'field crops.')

##### Cell 6 — Back up existing models

Copies current weights to `*_prev.pth` before overwriting.

**To roll back:** `shutil.copy(MODEL_DIR/'binary_prev.pth', MODEL_DIR/'binary_best.pth')`

In [ ]:
from datetime import datetime as _dt
import shutil
# Timestamped backup so each retrain keeps its own snapshot
_bk_ts = _dt.now().strftime('%Y%m%d_%H%M%S')
backups = [
    ('binary_best.pth',         f'binary_best_{_bk_ts}.pth'),
    ('5group_efficientnet.pth', f'5group_efficientnet_{_bk_ts}.pth'),
    ('4group_insectnet.pth',    f'4group_insectnet_{_bk_ts}.pth'),
    ('5group_insectnet.pth',    f'5group_insectnet_{_bk_ts}.pth'),
]
for src_name, dst_name in backups:
    src = MODEL_DIR/src_name; dst = MODEL_DIR/dst_name
    if src.exists(): shutil.copy(str(src),str(dst)); print(f'  Backed up: {src_name} -> {dst_name}')
    else: print(f'  Skipped (not found): {src_name}')
print('\nBackup complete. Safe to retrain.')

##### Cell 7 — Retrain binary classifier

Fine-tunes `binary_best.pth`. Best checkpoint saved to `RUN_DIR/` then copied back to `models/`.
Skip by setting `RETRAIN_BINARY = False` in Cell 2.

In [ ]:
if not RETRAIN_BINARY:
    print('RETRAIN_BINARY=False — skipping.')
else:
    print('Loading binary classifier for fine-tuning...')
    model_bin = load_efficientnet_for_finetune(
        BINARY_MODEL, n_classes=2, freeze_backbone=FREEZE_BACKBONE).to(DEVICE)
    tr_ldr_bin = make_loader(binary_data, bin_tr, IMG_SIZE_BIN, BATCH, aug=True)
    va_ldr_bin = make_loader(binary_data, bin_va, IMG_SIZE_BIN, BATCH, weighted=False)
    model_bin = run_finetune(model_bin,'binary_retrain',tr_ldr_bin,va_ldr_bin,
                             EPOCHS_BINARY,LR_FINETUNE,bin_counts,RUN_DIR/'binary_best.pth',
                             DEVICE,CLASSES_BINARY,RUN_DIR,IN_COLAB,
                             best_metric='recall_insect', img_size=IMG_SIZE_BIN)
    shutil.copy(RUN_DIR/'binary_best.pth', MODEL_DIR/'binary_best.pth')
    print(f'  → models/binary_best.pth updated from {RUN_DIR.name}')
    te_ldr_bin = make_loader(binary_data, bin_te, IMG_SIZE_BIN, BATCH, weighted=False)
    te_bin = eval_epoch(model_bin,te_ldr_bin,weighted_criterion(bin_counts,2,DEVICE),DEVICE,CLASSES_BINARY)
    print(f'\nBinary test: MacroF1={te_bin["macro_f1"]:.3f}  Acc={te_bin["acc"]:.3f}')
    if HAS_SKLEARN: print(classification_report(te_bin['labels'],te_bin['preds'],labels=list(range(len(CLASSES_BINARY))),target_names=CLASSES_BINARY,digits=3,zero_division=0))


##### Cell 8 — Retrain 5-class EfficientNet-B2

Fine-tunes `5group_efficientnet.pth` (EfficientNet-B2 backbone).
Best checkpoint saved to `RUN_DIR/` then copied back to `models/`.
Skip by setting `RETRAIN_5CLASS_EFFNET = False` in Cell 2.

In [ ]:
if not RETRAIN_5CLASS_EFFNET:
    print('RETRAIN_5CLASS_EFFNET=False — skipping.')
else:
    print('Loading 5-class EfficientNet for fine-tuning...')
    model_5cls = load_efficientnet_for_finetune(
        MODEL_5CLASS_EFFNET, n_classes=5, freeze_backbone=FREEZE_BACKBONE).to(DEVICE)
    tr_ldr_5 = make_loader(smp_5, tr_5, IMG_SIZE_5CLS_EN, BATCH, aug=True)
    va_ldr_5 = make_loader(smp_5, va_5, IMG_SIZE_5CLS_EN, BATCH, weighted=False)
    model_5cls = run_finetune(model_5cls,'5class_efficientnet_retrain',tr_ldr_5,va_ldr_5,
                              EPOCHS_5CLS_EN,LR_FINETUNE,counts_5,RUN_DIR/'5group_efficientnet.pth',
                              DEVICE,CLASSES_5,RUN_DIR,IN_COLAB,
                              best_metric='macro_f1', img_size=IMG_SIZE_5CLS_EN)
    shutil.copy(RUN_DIR/'5group_efficientnet.pth', MODEL_DIR/'5group_efficientnet.pth')
    print(f'  → models/5group_efficientnet.pth updated from {RUN_DIR.name}')
    te_ldr_5 = make_loader(smp_5, te_5, IMG_SIZE_5CLS_EN, BATCH, weighted=False)
    te_5cls = eval_epoch(model_5cls,te_ldr_5,weighted_criterion(counts_5,5,DEVICE),DEVICE,CLASSES_5)
    print(f'\n5-class EfficientNet test: MacroF1={te_5cls["macro_f1"]:.3f}  Acc={te_5cls["acc"]:.3f}')
    if HAS_SKLEARN: print(classification_report(te_5cls['labels'],te_5cls['preds'],labels=list(range(len(CLASSES_5))),target_names=CLASSES_5,digits=3,zero_division=0))


##### Cell 8b — Retrain 5-class InsectNet

Fine-tunes `5group_insectnet.pth` (InsectNet / RegNet-Y-32GF backbone) using the same
5-class data as Cell 8. Best checkpoint saved to `RUN_DIR/` then copied back to `models/`.
Skip by setting `RETRAIN_5CLASS_INSECTNET = False` in Cell 2.

In [ ]:
CLASSES_5_IN = ['bumblebee', 'fly', 'butterfly', 'other', 'background']
ALIAS_5_IN   = {
    'bumblebee':      'bumblebee',
    'fly':            'fly',
    'butterfly':      'butterfly',
    'butterfly_moth': 'butterfly',
    'other':          'other',
    'background':     'background',
}

if not RETRAIN_5CLASS_INSECTNET:
    print('RETRAIN_5CLASS_INSECTNET=False — skipping.')
else:
    print('Loading 5-class InsectNet for fine-tuning...')
    model_5cls_in = load_insectnet_for_finetune(
        MODEL_5CLASS_INSECTNET, n_classes=5,
        freeze_backbone=FREEZE_BACKBONE).to(DEVICE)

    # Reuse the same 5-class data splits already computed in Cell 5
    # (smp_5 / tr_5 / va_5 / te_5 / counts_5 — same labels, different model)
    if len(smp_5) == 0:
        print('WARNING: no 5-class crops found — check DATASET_DIRS.')
    else:
        tr_ldr_5in = make_loader(smp_5, tr_5, IMG_SIZE_5CLS_IN, BATCH, aug=True)
        va_ldr_5in = make_loader(smp_5, va_5, IMG_SIZE_5CLS_IN, BATCH, weighted=False)

        model_5cls_in = run_finetune(
            model_5cls_in, '5class_insectnet_retrain',
            tr_ldr_5in, va_ldr_5in,
            EPOCHS_5CLS_IN, LR_FINETUNE, counts_5,
            RUN_DIR / '5group_insectnet.pth',
            DEVICE, CLASSES_5, RUN_DIR, IN_COLAB,
            best_metric='macro_f1', img_size=IMG_SIZE_5CLS_IN)
        shutil.copy(RUN_DIR / '5group_insectnet.pth', MODEL_DIR / '5group_insectnet.pth')
        print(f'  → models/5group_insectnet.pth updated from {RUN_DIR.name}')

        # Test evaluation
        te_ldr_5in = make_loader(smp_5, te_5, IMG_SIZE_5CLS_IN, BATCH, weighted=False)
        ta_5in = eval_epoch(model_5cls_in, te_ldr_5in,
                            torch.nn.CrossEntropyLoss(), DEVICE, CLASSES_5)
        print(f'\n5-class InsectNet test: MacroF1={ta_5in["macro_f1"]:.3f}  Acc={ta_5in["acc"]:.3f}')
        if HAS_SKLEARN:
            from sklearn.metrics import classification_report
            print(classification_report(ta_5in['labels'], ta_5in['preds'],
                                        labels=list(range(len(CLASSES_5))),
                                        target_names=CLASSES_5, digits=3, zero_division=0))


##### Cell 9 — Retrain 4-class InsectNet

Fine-tunes `4group_insectnet.pth` (InsectNet / RegNet-Y-32GF backbone) on the new labeled crops.
Best checkpoint saved to `RUN_DIR/` then copied back to `models/4group_insectnet.pth`.
Skip by setting `RETRAIN_4CLASS = False` in Cell 2.

Only the `fc` head is updated when `FREEZE_BACKBONE = True` (recommended for small new datasets).
Set `FREEZE_BACKBONE = False` to also unfreeze `trunk_output.block4` for larger updates.


In [ ]:
CLASSES_4 = ['bumblebee', 'fly', 'butterfly', 'other']
ALIAS_4   = {
    'bumblebee': 'bumblebee', 'fly': 'fly',
    'butterfly': 'butterfly', 'butterfly_moth': 'butterfly',
    'other': 'other',
}

if not RETRAIN_4CLASS:
    print('RETRAIN_4CLASS=False — skipping.')
else:
    print('Loading 4-class InsectNet for fine-tuning...')
    model_4cls = load_insectnet_for_finetune(
        MODEL_4CLASS, n_classes=4,
        freeze_backbone=FREEZE_BACKBONE).to(DEVICE)

    # Collect 4-class data (insect crops only, no background)
    smp_4, counts_4 = collect_crops(DATASET_DIRS, CLASSES_4, ALIAS_4)
    print(f'\n4-class samples: {len(smp_4)}')
    for i, c in enumerate(CLASSES_4):
        print(f'  {c}: {counts_4.get(i, 0)}')

    if len(smp_4) == 0:
        print('WARNING: no 4-class crops found — check DATASET_DIRS and folder names.')
    else:
        tr_4, va_4, te_4 = stratified_split(smp_4)
        tr_ldr_4 = make_loader(smp_4, tr_4, IMG_SIZE_4CLS, BATCH, aug=True)
        va_ldr_4 = make_loader(smp_4, va_4, IMG_SIZE_4CLS, BATCH, weighted=False)

        model_4cls = run_finetune(
            model_4cls, '4class_insectnet_retrain',
            tr_ldr_4, va_ldr_4,
            EPOCHS_4CLASS, LR_FINETUNE, counts_4,
            RUN_DIR / '4group_insectnet.pth',
            DEVICE, CLASSES_4, RUN_DIR, IN_COLAB,
            best_metric='macro_f1', img_size=IMG_SIZE_4CLS)
        shutil.copy(RUN_DIR / '4group_insectnet.pth', MODEL_DIR / '4group_insectnet.pth')
        print(f'  → models/4group_insectnet.pth updated from {RUN_DIR.name}')

        # Test evaluation
        te_ldr_4 = make_loader(smp_4, te_4, IMG_SIZE_4CLS, BATCH, weighted=False)
        ta = eval_epoch(model_4cls, te_ldr_4,
                        torch.nn.CrossEntropyLoss(), DEVICE, CLASSES_4)
        print(f'\nTest: MacroF1={ta["macro_f1"]:.3f}  Acc={ta["acc"]:.3f}')
        if HAS_SKLEARN:
            from sklearn.metrics import classification_report
            print(classification_report(ta['labels'], ta['preds'],
                                        labels=list(range(len(CLASSES_4))),
                                        target_names=CLASSES_4, digits=3, zero_division=0))


##### Cell 10 — Summary
Prints final model paths and next steps after retraining completes.

In [ ]:
print('='*55)
print('RETRAIN COMPLETE')
print('='*55)
print(f'\nRun dir : {RUN_DIR}')
updated = []
if RETRAIN_BINARY: updated.append('binary_best.pth')
if RETRAIN_5CLASS_EFFNET: updated.append('5group_efficientnet.pth')
if RETRAIN_4CLASS: updated.append('4group_insectnet.pth')
if RETRAIN_5CLASS_INSECTNET: updated.append('5group_insectnet.pth')
if updated:
    print('\nUpdated models:')
    for name in updated:
        p = MODEL_DIR/name; size_mb = p.stat().st_size/1e6 if p.exists() else 0
        print(f'  ✓ {name:<40} ({size_mb:.1f} MB)')
    print('\nBackups saved in models/ as:')
    for name in updated:
        stem = name.replace('.pth', '')
        # find matching timestamped backup
        matches = sorted(MODEL_DIR.glob(f'{stem}_????????_??????.pth'), reverse=True)
        if matches: print(f'  ✓ {matches[0].name}')
print('\nNext steps:')
print('  1. Run infer_cropbased.ipynb with a new RUN_NAME to test the updated models')
print('  2. Run evaluate.ipynb to compare new vs previous results')
print('  3. If improved: delete the timestamped *_YYYYMMDD_HHMMSS.pth backups from models/')
print('  4. If worse, roll back by copying the timestamped backup back:')
print('       import shutil')
print('       # e.g. shutil.copy(MODEL_DIR/"binary_best_20260529_140000.pth", MODEL_DIR/"binary_best.pth")')